# Experiments 8–11 — Parallel execution

Runs the four load experiments in one notebook, one experiment after the other, each with `MAX_WORKERS` worker processes. All configuration (traffic generation, algorithms, seeds, $N$ and $K$) comes from `_off_nominal_load_common.py`.

* Only **missing** replications are simulated. Rows already in each `replication_metrics.csv` are skipped, so the notebook can be interrupted and run again. Experiment 8 already contains the TD-only rows, which are not affected by the strategic terminal rule.
* Metrics are appended to the CSV every `SAVE_EVERY` replications and when a cell stops, so an interruption loses at most a few replications.
* Do not run this notebook and one of the individual experiment notebooks at the same time: both would append to the same CSV.
* Analysis and figures are produced afterwards by `Experiment 8-11 - Results and Plots.ipynb`.


In [1]:
from pathlib import Path
from time import perf_counter

from _off_nominal_load_common import (
    K,
    N_VALUES,
    STRATEGY_LABELS,
    execute_experiment_parallel,
    read_replication_metrics,
)

MAX_WORKERS = 8
KEEP_DATASETS = False  # only replication 1 of each case is kept on disk
OUTPUT_ROOT = Path("outputs")

# experiment label -> (scenario, strategies)
EXPERIMENTS = {
    "8": ("nominal", ("sd_only", "td_only", "sd_td")),
    "9": ("broad_wind", ("sd_only", "sd_td")),
    "10": ("geofences", ("sd_only", "sd_td")),
    "11": ("combined", ("sd_only", "sd_td")),
}


def run_experiment(experiment):
    """Run the missing replications of one experiment in parallel."""
    scenario, strategies = EXPERIMENTS[experiment]
    output_directory = OUTPUT_ROOT / f"output_{experiment}"
    print(f"Experiment {experiment} ({scenario}) -> {output_directory}")
    started_at = perf_counter()
    rows = execute_experiment_parallel(
        scenario=scenario,
        experiment_label=experiment,
        output_directory=output_directory,
        keep_datasets=KEEP_DATASETS,
        strategies=strategies,
        n_values=N_VALUES,
        replications=K,
        max_workers=MAX_WORKERS,
    )
    print(f"Experiment {experiment}: {len(rows)} rows available, "
          f"{(perf_counter() - started_at) / 60:.1f} min in this call.\n")
    return rows


print("Fleet sizes:", N_VALUES)
print("Replications per strategy and N:", K)
print("Workers:", MAX_WORKERS)


Fleet sizes: (5, 10, 15, 20, 25, 30, 35, 40, 45, 50)
Replications per strategy and N: 800
Workers: 8


## Experiment 8 — Nominal conditions

Only SD only and SD + TD are missing; TD only is already complete.


In [2]:
results_8 = run_experiment("8")


Experiment 8 (nominal) -> outputs/output_8
Running 16000 missing replications with 8 worker processes.
Completed 16000 new replications in 5501.0 s.
Saved: outputs/output_8/replication_metrics.csv
Experiment 8: 24000 rows available, 91.7 min in this call.



## Experiment 9 — Broad wind regions


In [3]:
results_9 = run_experiment("9")


Experiment 9 (broad_wind) -> outputs/output_9
Running 16000 missing replications with 8 worker processes.
Completed 16000 new replications in 6144.8 s.
Saved: outputs/output_9/replication_metrics.csv
Experiment 9: 16000 rows available, 102.4 min in this call.



## Experiment 10 — Dynamic geofences


In [4]:
results_10 = run_experiment("10")


Experiment 10 (geofences) -> outputs/output_10
Running 16000 missing replications with 8 worker processes.
Completed 16000 new replications in 6147.1 s.
Saved: outputs/output_10/replication_metrics.csv
Experiment 10: 16000 rows available, 102.5 min in this call.



## Experiment 11 — Combined conditions


In [5]:
results_11 = run_experiment("11")


Experiment 11 (combined) -> outputs/output_11
Running 16000 missing replications with 8 worker processes.
Completed 16000 new replications in 6416.9 s.
Saved: outputs/output_11/replication_metrics.csv
Experiment 11: 16000 rows available, 106.9 min in this call.



## Completeness check

Every strategy and fleet size should report $K$ replications.


In [6]:
for experiment, (_scenario, strategies) in EXPERIMENTS.items():
    path = OUTPUT_ROOT / f"output_{experiment}" / "replication_metrics.csv"
    rows = read_replication_metrics(path)
    print(f"Experiment {experiment}:")
    for strategy in strategies:
        counts = {
            n: len({row["replication"] for row in rows
                    if row["strategy"] == strategy and row["number_of_uavs"] == n})
            for n in N_VALUES
        }
        status = "complete" if set(counts.values()) == {K} else f"incomplete {counts}"
        print(f"  {STRATEGY_LABELS[strategy]:7s}: {status}")


Experiment 8:
  SD only: complete
  TD only: complete
  SD + TD: complete
Experiment 9:
  SD only: complete
  SD + TD: complete
Experiment 10:
  SD only: complete
  SD + TD: complete
Experiment 11:
  SD only: complete
  SD + TD: complete
